# 3rd Model: Hybrid NLP Ensemble for LLM Text Detection

This notebook implements the **3rd Model** approach for detecting AI-generated vs. Human-authored essays using **all available datasets**.

### Key Highlights:
- **Dataset Integration**: Combines `train_essays.csv`, `train_drcat_01.csv`, `LLM_generated_essay_PaLM.csv`, `falcon_180b_v1.csv`, and `llama_70b_v1.csv` (~350,000+ texts).
- **Feature Extraction**: Dual sublinear TF-IDF vectorization (Word 1-3 n-grams + Character-wb 3-5 n-grams).
- **Classifier Stack**: Soft Voting Ensemble combining `MultinomialNB` (probabilistic) and `SGDClassifier` (`modified_huber` loss).
- **Robustness**: High resilience against spelling noise and synthetic human-like errors introduced in hidden test sets.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score

print("Libraries loaded successfully.")

In [ ]:
# Step 1: Loading All Available Datasets
actual_dir = './Actual-Datasets'
extra_dir = './Extra-Datasets-Used'

dfs = []

# 1. Actual Train Dataset
train_actual_path = os.path.join(actual_dir, 'train_essays.csv')
if os.path.exists(train_actual_path):
    df_actual = pd.read_csv(train_actual_path)[['text', 'generated']].rename(columns={'generated': 'label'})
    dfs.append(df_actual)

# 2. Extra Datasets
for fn in ['train_drcat_01.csv', 'LLM_generated_essay_PaLM.csv', 'falcon_180b_v1.csv', 'llama_70b_v1.csv']:
    fp = os.path.join(extra_dir, fn)
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        if 'generated_text' in df.columns:
            df = df.rename(columns={'generated_text': 'text'})
        if 'generated' in df.columns:
            df = df.rename(columns={'generated': 'label'})
        if 'label' not in df.columns:
            df['label'] = 1
        dfs.append(df[['text', 'label']])

df_combined = pd.concat(dfs, ignore_index=True).dropna(subset=['text', 'label'])
df_combined['label'] = df_combined['label'].astype(int)
df_combined = df_combined.drop_duplicates(subset=['text'])

print(f"Total Combined Dataset Size: {len(df_combined)} samples")
print(f"Class Distribution: Human (0): {(df_combined['label']==0).sum()}, AI (1): {(df_combined['label']==1).sum()}")

In [ ]:
# Step 2: Feature Extraction using Word & Char TF-IDF
df_test = pd.read_csv(os.path.join(actual_dir, 'test_essays.csv'))

word_vec = TfidfVectorizer(ngram_range=(1, 3), lowercase=True, sublinear_tf=True, max_features=60000, strip_accents='unicode')
char_vec = TfidfVectorizer(ngram_range=(3, 5), analyzer='char_wb', lowercase=True, sublinear_tf=True, max_features=60000)

X_train_word = word_vec.fit_transform(df_combined['text'])
X_test_word = word_vec.transform(df_test['text'])

X_train_char = char_vec.fit_transform(df_combined['text'])
X_test_char = char_vec.transform(df_test['text'])

X_train = hstack([X_train_word, X_train_char]).tocsr()
X_test = hstack([X_test_word, X_test_char]).tocsr()
y_train = df_combined['label'].values

print(f"Feature matrix shape: {X_train.shape}")

In [ ]:
# Step 3: Model Building & Stratified K-Fold Validation
mnb = MultinomialNB(alpha=0.02)
sgd = SGDClassifier(loss='modified_huber', max_iter=5000, tol=1e-4, random_state=42)

ensemble = VotingClassifier(estimators=[('mnb', mnb), ('sgd', sgd)], voting='soft', weights=[0.3, 0.7])

# 5-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_indices = np.random.RandomState(42).choice(len(df_combined), size=min(30000, len(df_combined)), replace=False)
X_cv, y_cv = X_train[cv_indices], y_train[cv_indices]

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_cv, y_cv)):
    ensemble.fit(X_cv[trn_idx], y_cv[trn_idx])
    probs = ensemble.predict_proba(X_cv[val_idx])[:, 1]
    auc = roc_auc_score(y_cv[val_idx], probs)
    print(f"Fold {fold+1} ROC-AUC: {auc:.5f}")

In [ ]:
# Step 4: Final Training & Submission Generation
ensemble.fit(X_train, y_train)
preds = ensemble.predict_proba(X_test)[:, 1]

df_test['generated'] = preds
df_test[['id', 'generated']].to_csv('submission_third_model.csv', index=False)
df_test[['id', 'generated']].to_csv('submission.csv', index=False)
print("Submission successfully created:")
print(df_test[['id', 'generated']])